In [ ]:
pip install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from ucimlrepo import fetch_ucirepo

# ==========================================
# 1. Load Dataset (9 FEATURE VERSION)
# ==========================================
breast_cancer_wisconsin_original = fetch_ucirepo(id=15)

X = breast_cancer_wisconsin_original.data.features
y = breast_cancer_wisconsin_original.data.targets

# Convert to numeric (important)
X = X.apply(pd.to_numeric, errors='coerce')
y = y.apply(pd.to_numeric, errors='coerce')

# Remove missing values
combined_df = pd.concat([X, y], axis=1).dropna()

X = combined_df[X.columns]
y = combined_df[y.columns].squeeze().astype(int)

# Convert labels: 2 → 0, 4 → 1
y = y.replace({2: -1, 4: 1})

print("Dataset shape:", X.shape)
print("Labels:", np.unique(y))

# ==========================================
# 2. Train-Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42
)

# ==========================================
# 3. Min-Max Normalization
# ==========================================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================
# 4. Train SVM with RBF Kernel
# ==========================================
clf = svm.SVC(
    kernel='rbf',
    gamma=1/X_train.shape[1],
    C=1.0
)

clf.fit(X_train, y_train)

print("Training completed")
print("Number of support vectors:", clf.support_vectors_.shape[0])

# ==========================================
# 5. Select ONLY ONE Test Sample
# ==========================================
sample_index = 0

x_single = X_test[sample_index]
y_single = y_test[sample_index]

decision_score = clf.decision_function([x_single])

print("Ground truth:", y_single)
print("decisions score:", decision_score)
# ==========================================
# 6. Save Model Parameters
# ==========================================
np.savetxt("support_vectors_rbf.txt", clf.support_vectors_)
np.savetxt("dual_coeff_rbf.txt", clf.dual_coef_)
np.savetxt("intercept_rbf.txt", clf.intercept_)

np.savetxt("xtest_rbf.txt", x_single)
np.savetxt("ytest_rbf.txt", [y_single])
np.savetxt("yclassificationscore.txt", decision_score)

print("All files saved successfully.")

Dataset shape: (683, 9)
Labels: [-1  1]
Training completed
Number of support vectors: 64
Ground truth: 1
decisions score: [1.19306836]
All files saved successfully.


In [ ]:
# ==========================================
# Test dataset display
# ==========================================

print("test datasets")
for i in range(len(X_test)):
    print(X_test[i], "\t", y_test[i], "\n")

x_test_df = pd.DataFrame(X_test)
x_test_df['target'] = y_test

print("Test dataset:")
print(x_test_df)

test datasets
[0.77777778 0.22222222 0.33333333 0.88888889 0.22222222 1.
 0.22222222 0.22222222 0.        ] 	 1 

[0.77777778 0.77777778 0.66666667 0.33333333 1.         1.
 0.66666667 0.77777778 0.66666667] 	 1 

[0.         0.         0.         0.         0.11111111 0.
 0.22222222 0.         0.        ] 	 -1 

[0.         0.         0.11111111 0.11111111 0.11111111 0.
 0.22222222 0.         0.        ] 	 -1 

[0.         0.         0.         0.         0.         0.
 0.11111111 0.         0.        ] 	 -1 

[0.44444444 0.         0.         0.         0.22222222 0.11111111
 0.11111111 0.11111111 0.        ] 	 -1 

[0.         0.         0.22222222 0.         0.         0.
 0.11111111 0.         0.        ] 	 -1 

[0.         0.         0.         0.11111111 0.         0.22222222
 0.         0.         0.66666667] 	 -1 

[0.33333333 0.11111111 0.22222222 0.44444444 0.22222222 0.77777778
 0.66666667 0.55555556 0.        ] 	 1 

[0.33333333 0.         0.11111111 0.         0.11111111 

In [ ]:
# ==========================================
# Evaluation
# ==========================================
from sklearn.metrics import classification_report, accuracy_score

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))
print("accuracy:", accuracy_score(y_test, y_pred))

from sklearn.metrics import recall_score, precision_score, f1_score

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

              precision    recall  f1-score   support

          -1       0.94      0.99      0.96        79
           1       0.98      0.91      0.95        58

    accuracy                           0.96       137
   macro avg       0.96      0.95      0.95       137
weighted avg       0.96      0.96      0.96       137

accuracy: 0.9562043795620438
Precision: 0.9814814814814815
Recall: 0.9137931034482759
F1-score: 0.9464285714285714
